<a href="https://colab.research.google.com/github/mustafemohamedmalinm-lgtm/bioinformatics-beginning-/blob/main/my_first_AI_genomic_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("GPU ma diyaar baa sxb?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Magaca GPU-da aad haysato:", torch.cuda.get_device_name(0))


PyTorch Version: 2.11.0+cu128
GPU ma diyaar baa sxb?: True
Magaca GPU-da aad haysato: Tesla T4


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Soo dhuuko xogta kansarka oo u diyaari qaab PyTorch u baahan yahay
data = load_breast_cancer()
X, y = data.data, data.target

# Aad bay u muhiim tahay in xogta la cabbiro (Standardize) ka hor Deep Learning
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# U beddel xogta "Tensors" (waa qaabka PyTorch u akhriyo xogta)
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.FloatTensor(y_train).unsqueeze(1)
y_test = torch.FloatTensor(y_test).unsqueeze(1)

# 2. Naqshadaynta Deep Learning Model-ka (ANN)
class CancerClassifier(nn.Module):
    def __init__(self, input_dim):
        super(CancerClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, 16) # Lakabka 1-aad (Hidden Layer)
        self.relu = nn.ReLU()                  # Function-ka firfircoonida
        self.layer2 = nn.Linear(16, 1)         # Lakabka Output-ka (Kansar / Caafimaad)
        self.sigmoid = nn.Sigmoid()            # Probability u beddelaya (0 ilaa 1)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

# Bilow moodalka
model = CancerClassifier(X_train.shape[1])
criterion = nn.BCELoss() # Sida lagu cabbiro khaladaadka (Loss function)
optimizer = optim.Adam(model.parameters(), lr=0.01) # Habka loo saxayo koodhka (Optimizer)

# 3. Tababarka Moodalka (Training Loop)
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward() # Dib u noqo oo sax khaladaadka (Backpropagation)
    optimizer.step()

# 4. Imtixaanka Moodalka (Evaluation)
model.eval()
with torch.no_grad():
    predictions = model(X_test)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y_test).float().mean()
    print(f"Saxsanaanta Deep Learning Model-kaaga sxb waa: {accuracy.item() * 100:.2f}%")


Saxsanaanta Deep Learning Model-kaaga sxb waa: 98.25%


In [ ]:
!pip install biopython

from Bio import Entrez, SeqIO

# 1. Had iyo jeer NCBI u sheeg email-kaaga si ay kuu aqoonsadaan (waa qasab)
Entrez.email = "magacaaga@example.com"

print("Hadda waxaan xogta ka soo dejisanaa kaydka NCBI... Fadlan sug sxb.")

# 2. Isticmaal Entrez.efetch si aad u soo dhuukato hidde-sidaha TP53 ee aadanaha
# 'id' waa lambarka aqoonsiga ee NCBI ee hiddesidaha TP53 (NC_000017.11)
# 'db' waa database-ka Nucleotide, 'rettype' na waa qaabka FASTA oo ah kan ugu caansan Genomics-ka
handle = Entrez.efetch(db="nucleotide", id="NC_000017.11", rettype="fasta", retmode="text")

# 3. Akhriso xogta oo u beddel qaab koodhka Python fahmi karo
record = SeqIO.read(handle, "fasta")
handle.close()

# 4. Ku kaydi faylka gudaha Google Colab si aad hadhow u isticmaasho
SeqIO.write(record, "TP53_human.fasta", "fasta")

print("--- Xogtii Waa La Soo Dejiyey Sxb! ---")
print(f"Magaca faylka: {record.id}")
print(f"Sharaxaadda: {record.description}")
print(f"Dhererka DNA-da (Bases): {len(record.seq)} pairs")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:00
Hadda waxaan xogta ka soo dejisanaa kaydka NCBI... Fadlan sug sxb.
--- Xogtii Waa La Soo Dejiyey Sxb! ---
Magaca faylka: NC_000017.11
Sharaxaadda: NC_000017.11 Homo sapiens chromosome 17, GRCh38.p14 Primary Assembly
Dhererka DNA-da (Bases): 83257441 pairs


import numpy as np
import torch

# 1. Soo qaado DNA-dii weynayd ee aad NCBI ka soo dejisay
# Aynu ka soo qaadano qayb weyn oo DNA-da ah si aan u helno xarfo 'ACGT' ah.
# Ka dib, ka tirtir dhammaan xarfooyinka 'N' ee aan la garanayn.
# Waxaan isticmaaleynaa 1,000,000 xarfo si aan u hubino in aan helno xog ku filan.
full_dna_raw = str(record.seq[:1000000]).upper()
full_dna = "".join(base for base in full_dna_raw if base in 'ACGT')

window_size = 100
dna_windows = []

# 2. Googooy DNA-da adoo isticmaalaya Sliding Window (mid walba waa 100 xarfo)
if len(full_dna) < window_size:
    print(f"Dhererka DNA-da ee la nadiifiyey ({len(full_dna)} bases) wuu ka yar yahay window_size ({window_size} bases).")
    print("Ma jirto qayb (window) oo la heli karo si loo sii wado. Fadlan kordhi qaybta DNA-da la soo qaadanayo (record.seq[:X]).")
    X_dna = torch.FloatTensor([]) # Initialize as empty tensor
else:
    for i in range(0, len(full_dna) - window_size + 1, window_size):
        window = full_dna[i:i+window_size]
        dna_windows.append(window)

    # Check if dna_windows is empty after processing
    if not dna_windows:
        print("Ma jirto wax DNA windows ah oo la sameeyay ka dib nadiifinta iyo googooynta.")
        X_dna = torch.FloatTensor([]) # Initialize as empty tensor
    else:
        # 3. Samee sharcigii One-Hot Encoding ee dhamaan qaybihii la gooyey
        mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
        encoded_windows = []

        for window in dna_windows:
            numerical_seq = [mapping[base] for base in window]
            one_hot = np.zeros((window_size, 4))
            for idx, base_idx in enumerate(numerical_seq):
                one_hot[idx, base_idx] = 1.0
            encoded_windows.append(one_hot)

        # 4. U beddel PyTorch Tensors
        X_dna = torch.FloatTensor(np.array(encoded_windows))

print("--- Xogtii DNA-da Waa La Googooyey Sxb! ---")
if X_dna.shape[0] > 0:
    print(f"Muxuu noqday tirada qaybaha (Windows) ee la helay?: {X_dna.shape[0]} qaybood")
    print(f"Xajmiga matrix-ka hal qayb kasta (Length, Channels): {X_dna[0].shape}")
    print(f"Xajmiga guud ee xogta (Batch, Length, Channels): {X_dna.shape}")
else:
    print("Wax DNA windows ah lama samayn. X_dna waa maran.")

In [ ]:
# 1. Hubi in model-ka cnn uu ku jiro habka qiimaynta (Evaluation mode)
model_cnn.eval()

# 2. Xogta u dhex rido model-ka adoo adeegsanaya torch.no_grad() si aanay memory-ga u culaysin
with torch.no_grad():
    predictions = model_cnn(X_dna)

print("--- Saadaashii GenomeCNN Waa Diyaar Sxb! ---")
print(f"Xajmiga Saadaasha (Predictions Shape): {predictions.shape}")
print("\n5-ta Saadaal ee ugu horreeya (Probability 0 ilaa 1):")
print(predictions[:5])


In [ ]:
import pandas as pd

# 1. Waxaan abuureynaa xog VCF ah oo moodal u ah 5 bukaan oo kansar qaba
vcf_data = """##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##INFO=<ID=GENE,Number=1,Type=String,Description="Gene Name">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
17\t7577121\trs12345\tG\tA\t100\tPASS\tDP=45;GENE=TP53
17\t7578263\trs67890\tC\tT\t95\tPASS\tDP=38;GENE=TP53
17\t7579472\t.\tA\tG\t50\tLowQual\tDP=12;GENE=TP53
17\t43044295\trs5555\tT\tC\t100\tPASS\tDP=60;GENE=BRCA1
17\t43045712\t.\tG\tC\t99\tPASS\tDP=55;GENE=BRCA1
"""

# 2. Ku kaydi xogtan fayl la yiraahdo 'cancer_mutations.vcf' gudaha Colab
with open("cancer_mutations.vcf", "w") as f:
    f.write(vcf_data)

# 3. Koodhka Python ee lagu akhrinayo faylka VCF adoo adeegsanaya Pandas
# Waxaan ka leexaneynaa xariiqyada ku bilaabma '##' sababtoo ah waa metadata
df_vcf = pd.read_csv("cancer_mutations.vcf", comment='#', sep='\t', header=None)
df_vcf.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']

print("--- Faylkii VCF Ee Kansarka Waa La Akhriyey Sxb! ---")
print(f"Tirada Mutations-ka la helay: {len(df_vcf)}")
print("\nXogta rasmiga ah ee VCF-ka:")
display(df_vcf)


--- Faylkii VCF Ee Kansarka Waa La Akhriyey Sxb! ---
Tirada Mutations-ka la helay: 5

Xogta rasmiga ah ee VCF-ka:


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,17,7577121,rs12345,G,A,100,PASS,DP=45;GENE=TP53
1,17,7578263,rs67890,C,T,95,PASS,DP=38;GENE=TP53
2,17,7579472,.,A,G,50,LowQual,DP=12;GENE=TP53
3,17,43044295,rs5555,T,C,100,PASS,DP=60;GENE=BRCA1
4,17,43045712,.,G,C,99,PASS,DP=55;GENE=BRCA1


In [ ]:
# 1. Kala soo bax magaca Gene-ka adoo isticmaalaya Regex (sharciga raadinta qoraalka)
df_vcf['GENE_NAME'] = df_vcf['INFO'].str.extract(r'GENE=([A-Za-z0-9]+)')

# 2. Sifee xogta: Kaliya reeb mutations-ka tayadoodu sarreyso (PASS)
# Sidoo kale, ka dooro kaliya hidde-sidaha rasmiga ah ee TP53
nadiif_df = df_vcf[(df_vcf['FILTER'] == 'PASS') & (df_vcf['GENE_NAME'] == 'TP53')]

print("--- Xogtii VCF Waa La Sifeyey Sxb! ---")
print(f"Mutations-ka tayada leh ee TP53 waa: {len(nadiif_df)} safood")
print("\nXogta nadiifta ah ee hadda u diyaar ah Machine Learning:")
display(nadiif_df[['CHROM', 'POS', 'REF', 'ALT', 'QUAL', 'GENE_NAME']])


--- Xogtii VCF Waa La Sifeyey Sxb! ---
Mutations-ka tayada leh ee TP53 waa: 2 safood

Xogta nadiifta ah ee hadda u diyaar ah Machine Learning:


,CHROM,POS,REF,ALT,QUAL,GENE_NAME
0,17,7577121,G,A,100,TP53
1,17,7578263,C,T,95,TP53


In [ ]:
import numpy as np
import torch

# 1. Soo qaado mid ka mid ah boosaska (POS) ee shaxda nadiifta ah ku jira
# Tusaale: Waxaan rabaa inaan eego mutation-ka koowaad ee booskiisu yahay 7577121
mutation_pos = 7577121

# Maadaama silsiladdii DNA ee aan soo dejisay ay ka bilaabato eber, aynu ka soo qaadno
# inaan soo gooyneyno 10 xarfo oo ka horreeya booska iyo 10 xarfo oo ka dambeeya (Flanking Region)
# Si koodhku Colab ugu ordo, aynu tusaale ahaan ka soo qaadno booskaas inuu ku jiro xogtii aan haynay
start_idx = 50000  # Meeshii aan koodhkii hore ka bilownay
end_idx = 50020

# 2. Soo goos aagga mutation-ku ku hareeraysan yahay (Window sequence)
mutated_window = str(record.seq[start_idx:end_idx]).upper()

# 3. Samee isbeddelka dhabta ah: Beddel xarafka caadiga ah (REF: G) kuna beddel kan kansarka (ALT: A)
# Waxaan tusaale u soo qaadanaynaa in xarafka dhexe uu isbeddelayo
window_list = list(mutated_window)
mid_point = len(window_list) // 2
window_list[mid_point] = 'A'  # Halkii ay 'G' ka ahayd, hadda waa 'A' (Cancer Mutation)
mutated_sequence = "".join(window_list)

# 4. U beddel One-Hot Encoding si uu CNN-ka u akhriyo
mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
numerical_seq = [mapping[base] for base in mutated_sequence if base in 'ACGT']

one_hot_mutated = np.zeros((len(numerical_seq), 4))
for idx, base_idx in enumerate(numerical_seq):
    one_hot_mutated[idx, base_idx] = 1.0

# 5. U beddel PyTorch Tensor oo diyaari xajmiga (Batch, Length, Channels)
final_input = torch.FloatTensor(one_hot_mutated).unsqueeze(0)

print("--- Matrix-kii Mutation-ka WES Waa Diyaar Sxb! ---")
print(f"Silsiladdii DNA-da ee isbedeshay: {mutated_sequence}")
print(f"Xajmiga loo dhex ridayo AI-da: {final_input.shape}")


--- Matrix-kii Mutation-ka WES Waa Diyaar Sxb! ---
Silsiladdii DNA-da ee isbedeshay: NNNNNNNNNNANNNNNNNNN
Xajmiga loo dhex ridayo AI-da: torch.Size([1, 1, 4])


In [ ]:
import numpy as np
import torch

# 1. Waxaan dhex raadinaynaa chromosome 17 meel nadiif ah oo dhererkeedu yahay 20 xarfo
# Meeshaas waa inaanay ku jirin wax 'N' ah gabi ahaanba
chromosome_seq = str(record.seq)
nadiif_start = 0

for i in range(100000, len(chromosome_seq) - 20):
    window_check = chromosome_seq[i:i+20].upper()
    if 'N' not in window_check:
        nadiif_start = i
        mutated_window = window_check
        break

print(f"Waxaan helay aag nadiif ah oo ka bilaabma index: {nadiif_start}")
print(f"Silsiladdii caadiga ahayd (Reference): {mutated_window}")

# 2. Samee isbeddelka dhabta ah: Beddel xarafka dhexe kuna beddel 'A' (Cancer Mutation)
window_list = list(mutated_window)
mid_point = len(window_list) // 2
window_list[mid_point] = 'A'  # Isbeddelkii kansarka
mutated_sequence = "".join(window_list)

# 3. U beddel One-Hot Encoding
mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
numerical_seq = [mapping[base] for base in mutated_sequence]

one_hot_mutated = np.zeros((len(numerical_seq), 4))
for idx, base_idx in enumerate(numerical_seq):
    one_hot_mutated[idx, base_idx] = 1.0

# 4. U beddel PyTorch Tensor (Batch, Length, Channels)
final_input = torch.FloatTensor(one_hot_mutated).unsqueeze(0)

print("\n--- Matrix-kii Mutation-ka WES Waa Diyaar Sxb! ---")
print(f"Silsiladdii DNA-da ee isbedeshay: {mutated_sequence}")
print(f"Xajmiga loo dhex ridayo AI-da: {final_input.shape}")


Waxaan helay aag nadiif ah oo ka bilaabma index: 100000
Silsiladdii caadiga ahayd (Reference): CTAATTTCTTTCATCACTAT

--- Matrix-kii Mutation-ka WES Waa Diyaar Sxb! ---
Silsiladdii DNA-da ee isbedeshay: CTAATTTCTTACATCACTAT
Xajmiga loo dhex ridayo AI-da: torch.Size([1, 20, 4])


In [ ]:
import torch
import torch.nn as nn

class GenomeCNN(nn.Module):
    def __init__(self):
        super(GenomeCNN, self).__init__()

        # 1. Conv1d: Wuxuu dhex marayaa silsiladda DNA-da (1D Convolution)
        # in_channels=4 (sababtoo ah waxaan haysanaa 4 xarfo A,C,G,T oo One-Hot ah)
        # out_channels=32 (wuxuu soo saarayaa 32 sifooyin oo hidde-sideyaal ah)
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=32, kernel_size=5, padding=2)
        self.relu = nn.ReLU()

        # 2. MaxPool1d: Wuxuu soo gaabinayaa xogta si uu muhiimada u soo saaro
        self.pool = nn.MaxPool1d(kernel_size=2)

        # 3. Fully Connected Layer (Lakabka Saadaasha)
        # Input-ku wuxuu ku xiran yahay dhererka qaybta DNA-da ee aad siiso
        self.fc = nn.Linear(32 * 50, 1)  # Tusaale haddii DNA-du dhererkeedu ahaa 100 base pairs
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # PyTorch Conv1d wuxuu u baahan yahay xogta qaabkan: (Batch, Channels, Length)
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.pool(x)

        # U beddel xogta hal dhaban (Flatten) ka hor lakabka ugu dambeeya
        x = x.view(x.size(0), -1)
        x = self.sigmoid(self.fc(x))
        return x

# Bilow model-ka cusub sxb
model_cnn = GenomeCNN()
print("--- Naqshaddii CNN-kaaga Genomics-ka Waa Diyaar! ---")
print(model_cnn)


--- Naqshaddii CNN-kaaga Genomics-ka Waa Diyaar! ---
GenomeCNN(
  (conv1): Conv1d(4, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1600, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
